# TP DBSCAN - Détection de zones géographiques NYC Taxi
## Solution complète avec VRAIES données


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import os
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# Configuration plots
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

# PARTIE 1 - PRÉPARATION DES DONNÉES

In [ ]:
print('='*80)
print('PARTIE 1 - PREPARATION DES DONNEES')
print('='*80)

# Etape 1: Charger VRAIES donnees NYC Taxi
print('\nChargement des VRAIES donnees NYC Taxi...')

# Chercher le fichier
fichier_csv = 'yellow_tripdata_2015-01.csv'

if not os.path.exists(fichier_csv):
    raise FileNotFoundError(f'ERREUR: Fichier {fichier_csv} non trouve!\nPlacez le fichier yellow_tripdata_2015-01.csv dans le meme dossier que ce notebook.')

print(f'OK: Fichier trouve: {fichier_csv}')

# Charger le fichier (gros fichier, lire en chunks ou limiter)
df = pd.read_csv(fichier_csv, nrows=50000)  # Limiter a 50k points pour perf
print(f'   - {len(df):,} points charges')
print(f'   - Colonnes: {list(df.columns)[:8]}...')

# Verifier colonnes GPS
if 'pickup_latitude' not in df.columns or 'pickup_longitude' not in df.columns:
    raise ValueError('ERREUR: Colonnes pickup_latitude ou pickup_longitude manquantes!')

print(f'\nStructure des donnees:')
print(df[['pickup_latitude', 'pickup_longitude']].head(10))
print(f'\nResume statistique (GPS):')
print(df[['pickup_latitude', 'pickup_longitude']].describe())

# Etape 2: Nettoyage
print('\n--- NETTOYAGE ---')
print(f'Avant: {len(df):,} points')

# Supprimer NULL
df = df.dropna(subset=['pickup_latitude', 'pickup_longitude'])

# Filtrer coordonnees invalides (0,0) et bounds NYC
nyc_bounds = {
    'lat_min': 40.50, 'lat_max': 40.95,
    'lon_min': -74.30, 'lon_max': -73.70
}

df = df[
    (df['pickup_latitude'] > 0) &
    (df['pickup_longitude'] != 0) &
    (df['pickup_latitude'] >= nyc_bounds['lat_min']) &
    (df['pickup_latitude'] <= nyc_bounds['lat_max']) &
    (df['pickup_longitude'] >= nyc_bounds['lon_min']) &
    (df['pickup_longitude'] <= nyc_bounds['lon_max'])
]

print(f'Apres nettoyage: {len(df):,} points')
print(f'\nBounds NYC filtres: {nyc_bounds}')

# Etape 3: Conversion en radians (pour Haversine)
coords_radians = np.radians(df[['pickup_latitude', 'pickup_longitude']].values)
print(f'\nOK: Donnees converties en radians pour distance Haversine')
print(f'  Premieres coordonnees: {coords_radians[0]}')

## Question : Pourquoi la distance euclidienne classique n'est pas idéale pour des coordonnées GPS ?

**RÉPONSE:**

La Terre est une **sphère**, pas un plan. La distance euclidienne suppose un espace 2D plat (projection), ce qui crée des erreurs massives avec les coordonnées GPS.

**EXEMPLE CONCRET:**
- Point A: Times Square (40.7128, -74.0060)
- Point B: 1 degré EST (40.7128, -73.0060)

Distance euclidienne: $\sqrt{(40.7128-40.7128)^2 + (-74.0060-(-73.0060))^2} = 1.0$ unites

Distance réelle (Haversine): **85 km** → **ERREUR = 85x!**

**INDICE: Distance sphérique (Haversine)** est obligatoire pour GPS.

# PARTIE 2 - COMPRÉHENSION PARAMETRES DBSCAN

In [ ]:
print('\n' + '='*80)
print('PARTIE 2 - QUESTIONS SUR LES PARAMETRES DBSCAN')
print('='*80)

print('''
QUESTION 1: Quelle unite doit avoir epsilon pour coordonnees GPS ?

REPONSE:
   Avec metric="haversine" dans sklearn:
   => epsilon est en RADIANS (pas en km!)
   
   TABLE CONVERSION:
   - 0.01 radian = 63.7 km
   - 0.001 radian = 6.37 km
   - 0.0001 radian = 637 m
   - 0.00005 radian = 318 m
   - 0.0003 radian ~= 1.9 km (petit hotspot)
   
   PRATIQUE pour hotspots NYC:
   - epsilon = 0.0005 radian ~= 3.2 km (hotspot concentre)
   - epsilon = 0.001 radian ~= 6.4 km (hotspot moyen)
   - epsilon = 0.002 radian ~= 12.8 km (zone large)

---

QUESTION 2: Pourquoi min_samples doit etre ajuste selon la densite urbaine ?

REPONSE:
   min_samples = nombre MINIMUM de points dans epsilon-voisinage
   pour qu'un point soit considere comme CORE POINT
   
   DENSITES URBAINES DIFFERENTES:
   
   - Zones residentielles (Brooklyn edge, Queens): ~10-50 pickups/km²
     => min_samples = 2-3 (tres sensible)
   
   - NYC moyen (neighborhoods, outer boroughs): ~100-300 pickups/km²
     => min_samples = 5-10 (recommande)
   
   - Manhattan/hotspots: ~1000+ pickups/km²
     => min_samples = 10-20+ (tres robuste)
   
   REGLE GENERALE:
   min_samples >= 2 * nb_dimensions
   Ici 2D => min_samples >= 4 minimum

---

QUESTION 3: Pourquoi la dimension 2D simplifie le choix des parametres ?

REPONSE:
   
   AVANTAGES DIMENSION 2D:
   - Seulement 2 parametres a tuner: epsilon et min_samples
   - Courbe k-distance facile a interpreter visuellement
   - PAS de "curse of dimensionality"
   - Densites spatiales plus uniformes
   - Visualisation DIRECTE sur carte (scatter 2D)
   - Intuition geometrique claire (rayon = epsilon)
   
   PROBLEMES HAUTE DIMENSION (10D+):
   - epsilon doit AUGMENTER exponentiellement
   - Tous les points deviennent "proches"
   - Resultat: 1 mega-cluster + bruit
   - Parametres completement impredictibles
   - Visualisation impossible
   - Densite se concentre en corners (effet de bord)
   
   EXEMPLE:
   - 2D: epsilon = 0.001 rad = 6.4 km => bonnes zones
   - 10D: epsilon = 0.001 rad => tout bruit, aucun cluster
             epsilon = 0.1 rad => 1 mega-cluster, tout fusionne
''')

# PARTIE 3 - CHOIX D'EPSILON (MÉTHODE k-DISTANCE)

In [ ]:
print('\n' + '='*80)
print('PARTIE 4 - METHODE k-DISTANCE POUR CHOISIR EPSILON')
print('='*80)

# Calculer distances au k-eme voisin
k = 5  # nombre de voisins

print(f'\nCalcul des distances au {k}-eme voisin (metrique Haversine)...')

nbrs = NearestNeighbors(n_neighbors=k, metric='haversine').fit(coords_radians)
distances, indices = nbrs.kneighbors(coords_radians)

# Le dernier voisin (index k-1)
distances = distances[:, k-1]
distances = np.sort(distances)[::-1]  # Ordre decroissant

# Identifier le "coude" visuellement
fig, ax = plt.subplots(figsize=(14, 6))

# Courbe complete
ax.plot(distances, 'b-', linewidth=1.5, label='k-distance')
ax.set_xlabel('Points tries (index)', fontsize=12)
ax.set_ylabel(f'Distance au {k}e voisin (radians)', fontsize=12)
ax.set_title(f'METHODE k-DISTANCE pour choisir epsilon\n(k={k}, metrique=haversine)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

# Identifier le coude automatiquement (difference de difference)
diff1 = np.diff(distances)
diff2 = np.diff(diff1)
coude_idx = np.argmax(diff2) + 1
epsilon_candidate = distances[coude_idx]

# Marquer le coude
ax.axhline(y=epsilon_candidate, color='red', linestyle='--', linewidth=2, label=f'Coude identifie (~{epsilon_candidate:.5f} rad)')
ax.plot(coude_idx, epsilon_candidate, 'ro', markersize=10, label='Coude')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f'\nCOUDE IDENTIFIE:')
print(f'   Epsilon candidate = {epsilon_candidate:.5f} radians')
print(f'   En km: {epsilon_candidate * 6371:.2f} km')
print(f'\n   Interpretation:')
print(f'   - Points AVANT coude: Clusters reels (denses)')
print(f'   - Points APRES coude: Bruit (isoles)')
print(f'   - Le coude separe les deux populations')

# Pour la suite, on utilise une valeur pratique
epsilon = 0.0005  # ~3.2 km, bon pour hotspots urbains
min_samples = 5

print(f'\nPARAMETRES CHOISIS POUR DBSCAN:')
print(f'   epsilon = {epsilon} radians (~{epsilon*6371:.1f} km)')
print(f'   min_samples = {min_samples}')
print(f'   metric = "haversine"')

# PARTIE 4 - APPLICATION DBSCAN

In [ ]:
print('\n' + '='*80)
print('PARTIE 5 - APPLICATION DBSCAN')
print('='*80)

# Appliquer DBSCAN avec metric='haversine'
dbscan = DBSCAN(eps=epsilon, min_samples=min_samples, metric='haversine')
clusters = dbscan.fit_predict(coords_radians)

# Ajouter au dataframe
df['cluster'] = clusters
df['cluster_name'] = df['cluster'].apply(lambda x: 'BRUIT' if x == -1 else f'Cluster {x}')

# Statistiques
n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_bruit = len(clusters[clusters == -1])

print(f'\nDBSCAN APPLIQUE:')
print(f'   Nombre de clusters: {n_clusters}')
print(f'   Nombre de points bruit: {n_bruit}')
print(f'   Nombre total de points: {len(df)}')
print(f'   Ratio bruit: {n_bruit/len(df)*100:.1f}%')

print(f'\n   Detail par cluster:')
for cluster_id in sorted(set(clusters)):
    if cluster_id == -1:
        label = 'BRUIT'
    else:
        label = f'Cluster {cluster_id}'
    
    count = len(clusters[clusters == cluster_id])
    print(f'     {label}: {count} points ({count/len(df)*100:.1f}%)')

# Centres des clusters
print(f'\nCENTRES DES CLUSTERS (en degres):')
for cluster_id in sorted(set(clusters)):
    if cluster_id == -1:
        continue
    
    mask = clusters == cluster_id
    center_lat = df[mask]['pickup_latitude'].mean()
    center_lon = df[mask]['pickup_longitude'].mean()
    
    print(f'   Cluster {cluster_id}: ({center_lat:.4f}, {center_lon:.4f})')

# PARTIE 5 - VISUALISATION CLUSTERS

In [ ]:
print('\n' + '='*80)
print('PARTIE 6 - VISUALISATION')
print('='*80)

# Creer figure
fig, ax = plt.subplots(figsize=(14, 10))

# Couleurs pour clusters
colors = plt.cm.tab10(np.linspace(0, 1, n_clusters))

# Plot clusters
for cluster_id in sorted(set(clusters)):
    if cluster_id == -1:
        # Bruit en rouge avec 'x'
        mask = clusters == cluster_id
        ax.scatter(df[mask]['pickup_longitude'], 
                  df[mask]['pickup_latitude'],
                  marker='x', color='red', s=100, 
                  label=f'Bruit ({len(df[mask])} pts)', alpha=0.6)
    else:
        # Cluster avec couleur
        mask = clusters == cluster_id
        color = colors[cluster_id]
        ax.scatter(df[mask]['pickup_longitude'], 
                  df[mask]['pickup_latitude'],
                  color=color, s=50, 
                  label=f'Cluster {cluster_id} ({len(df[mask])} pts)',
                  alpha=0.7, edgecolors='black', linewidth=0.3)

# Centres des clusters
for cluster_id in sorted(set(clusters)):
    if cluster_id == -1:
        continue
    
    mask = clusters == cluster_id
    center_lat = df[mask]['pickup_latitude'].mean()
    center_lon = df[mask]['pickup_longitude'].mean()
    
    ax.plot(center_lon, center_lat, 'k*', markersize=20, markeredgecolor='yellow', markeredgewidth=2)

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title(f'Zones geographiques identifiees par DBSCAN\nN={n_clusters} clusters, bruit={n_bruit} pts', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nVisualisation terminee')
print('   * = Centre du cluster')
print('   x = Points bruit')

# PARTIE 6 - INTERPRÉTATION MÉTIER

In [ ]:
print('\n' + '='*80)
print('PARTIE 7 - INTERPRETATION METIER DES ZONES')
print('='*80)

# Zones NYC representatives
zones_nyc = {
    'Manhattan Midtown': (40.758, -73.985),
    'Manhattan Downtown': (40.707, -74.011),
    'JFK Airport': (40.641, -73.778),
    'LaGuardia Airport': (40.777, -73.874),
    'Brooklyn Williamsburg': (40.706, -73.978),
    'Times Square': (40.758, -73.989),
    'Grand Central': (40.753, -73.976),
    'SoHo': (40.723, -74.002),
}

from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    """Calcule distance entre 2 points (resultat en km)"""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

print('\nINTERPRETATION PAR CLUSTER:\n')

for cluster_id in sorted(set(clusters)):
    if cluster_id == -1:
        continue
    
    mask = clusters == cluster_id
    center_lat = df[mask]['pickup_latitude'].mean()
    center_lon = df[mask]['pickup_longitude'].mean()
    n_points = len(df[mask])
    
    # Trouver zone la plus proche
    min_dist = float('inf')
    closest_zone = None
    for zone, (zone_lat, zone_lon) in zones_nyc.items():
        dist = haversine_km(center_lat, center_lon, zone_lat, zone_lon)
        if dist < min_dist:
            min_dist = dist
            closest_zone = zone
    
    print(f'CLUSTER {cluster_id}:')
    print(f'  Centre: ({center_lat:.4f}, {center_lon:.4f})')
    print(f'  Points: {n_points}')
    print(f'  Zone NYC la plus proche: {closest_zone} (dist={min_dist:.1f} km)')
    
    # Classification par type
    if 'Airport' in closest_zone:
        zone_type = 'ZONE AEROPORTUAIRE'
    elif any(x in closest_zone for x in ['Times Square', 'Grand Central', 'SoHo', 'Midtown']):
        zone_type = 'ZONE TOURISTIQUE/AFFAIRES (Manhattan)'
    elif 'Downtown' in closest_zone or 'Financial' in closest_zone:
        zone_type = 'ZONE AFFAIRES (Financial District)'
    elif 'Brooklyn' in closest_zone or 'Williamsburg' in closest_zone:
        zone_type = 'ZONE RESIDENTIELLE (Brooklyn)'
    else:
        zone_type = 'ZONE MIXTE'
    
    print(f'  Type: {zone_type}')
    print()

# PARTIE 7 - ANALYSE CRITIQUE

In [ ]:
print('\n' + '='*80)
print('PARTIE 8 - ANALYSE CRITIQUE')
print('='*80)

print('''
QUESTION 1: Que se passe-t-il si epsilon est TROP PETIT?

REPONSE:
   Si epsilon = 0.00001 radian (~63 metres):
   
   - Presque TOUS les points deviennent BRUIT
   - Aucun point n'a assez de voisins pour former cluster
   - Clusters fragmentes en micro-zones
   - Perte de structure globale
   - Peu interpretable metier
   - DBSCAN ne reconnait que hyper-concentrations
   
   RESULTAT: Peu d interet pour l analyse strategique
   Exemple: Chaque "coin de rue" = cluster separe

---

QUESTION 2: Que se passe-t-il si epsilon est TROP GRAND?

REPONSE:
   Si epsilon = 0.01 radian (~63 km):
   
   - TOUT fusionne en 1-2 mega-clusters
   - Tous les points NYC sont "proches" les uns des autres
   - Perte complete de distinction entre zones
   - Manhattan + Brooklyn + JFK = 1 mega-cluster
   - Peu de bruit (tout est classee)
   - Aucune information utile
   
   RESULTAT: Clustering inutile
   Exemple: "Manhattan = Brooklyn = Queens" c est un cluster

---

QUESTION 3: Que se passe-t-il si min_samples augmente fortement?

REPONSE:
   Si min_samples = 1000 au lieu de 5:
   
   - Clusters doivent avoir >=1000 voisins proches
   - Seuls hotspots HYPER-DENSES restent clusters
   - Beaucoup plus de points classes BRUIT
   - Nombre total de clusters diminue fortement
   - Clusters plus "purs" et concentres
   - Tolerance reduite pour dendrite/outliers
   
   EXEMPLE:
   - min_samples=5: 10 clusters + 500 bruit
   - min_samples=50: 4 clusters + 2000 bruit
   - min_samples=500: 1 cluster + 4000 bruit
   
   TRADE-OFF: Purete vs. couverture

---

QUESTION 4: DBSCAN peut-il detecter des DENSITES TRES DIFFERENTES?

REPONSE: NON - C EST UNE LIMITE MAJEURE
   
   Raison:
   - DBSCAN utilise le MEME epsilon PARTOUT
   - Manhattan: ~1000+ pickups/km² (hyper-densite)
   - Queens edge: ~50 pickups/km² (tres sparse)
   - Impossible satisfaire les deux avec 1 epsilon
   
   EXEMPLE CONCRET:
   - epsilon = 0.0005 rad (3.2 km) pour Manhattan
     => Queens fragmente en minuscules clusters
   
   - epsilon = 0.002 rad (12.8 km) pour Queens
     => Manhattan fusionne en mega-cluster
   
   CONSEQUENCE:
   DBSCAN classique ne gere PAS les densites heterogenes

---

QUESTION 5: Proposer ALTERNATIVE si densites heterogenes

REPONSES ALTERNATIVES:
   
   a) HDBSCAN (Hierarchical DBSCAN) - RECOMMANDE
      + Epsilon adaptatif par region
      + Detecte clusters a differentes echelles
      + Gere densite variable naturellement
      + Plus robuste sur donnees reelles
      - Plus complexe a interpreter
      - Plus lent calculatoirement
   
   b) Clustering hierarchique agglomeratif
      + Dendrogramme = structure complete
      + Coupure flexible selon profondeur
      + Gere bien heterogeneite
      - Tres lent sur gros datasets (>10k pts)
      - O(n^2) complexite spatiale
   
   c) GMM (Gaussian Mixture Model) via EM
      + Covariances differentes par cluster
      + Probabiliste (soft assignment vs. hard)
      + Suppose distributions gaussiennes
      + Rapide si nb_clusters connu
      - Peut etre piege dans optima locaux
      - Hypothese gaussienne pas toujours valide
   
   d) DBSCAN par zone + fusion post
      + Garder flexibilite DBSCAN
      + Pre-filtrer par BOROUGH (Manhattan vs Queens vs Brooklyn)
      + Parametres differents par zone
      + Fusionner clusters proches apres
      - Approche "artisanale"
      - Parametrisation manuelle complexe
   
   e) Affinity Propagation
      + Densities adaptatives
      + Pas de specification nombre clusters a priori
      - Lent O(n^2 * iterations)
      - Sensible au damping parameter
   
   RECOMMANDATION POUR NYC TAXI:
   => HDBSCAN pour robustesse et adaptativite optimales
''')

print('\n' + '='*80)
print('ANALYSE TERMINEE')
print('='*80)

# COMPARAISON: Effects of Parameter Variations

In [ ]:
print('\n' + '='*80)
print('DEMONSTRATION: EFFET DES PARAMETRES')
print('='*80)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Impact des parametres DBSCAN sur les clusters', fontsize=16, fontweight='bold')

# Cas 1: epsilon TROP PETIT
epsilon1 = 0.00005
db1 = DBSCAN(eps=epsilon1, min_samples=5, metric='haversine').fit_predict(coords_radians)
n_clusters1 = len(set(db1)) - (1 if -1 in db1 else 0)
n_bruit1 = len(db1[db1 == -1])

ax = axes[0, 0]
for cid in sorted(set(db1)):
    if cid == -1:
        mask = db1 == cid
        ax.scatter(df[mask]['pickup_longitude'], df[mask]['pickup_latitude'],
                  marker='x', color='red', s=30, alpha=0.6)
    else:
        mask = db1 == cid
        ax.scatter(df[mask]['pickup_longitude'], df[mask]['pickup_latitude'],
                  color=plt.cm.tab20(cid % 20), s=30, alpha=0.7)
ax.set_title(f'epsilon TROP PETIT ({epsilon1:.5f} rad)\n{n_clusters1} clusters, {n_bruit1} bruit ({n_bruit1/len(df)*100:.0f}%)', 
            fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)

# Cas 2: epsilon OPTIMAL (ce qu on a utilise)
epsilon2 = 0.0005
db2 = DBSCAN(eps=epsilon2, min_samples=5, metric='haversine').fit_predict(coords_radians)
n_clusters2 = len(set(db2)) - (1 if -1 in db2 else 0)
n_bruit2 = len(db2[db2 == -1])

ax = axes[0, 1]
for cid in sorted(set(db2)):
    if cid == -1:
        mask = db2 == cid
        ax.scatter(df[mask]['pickup_longitude'], df[mask]['pickup_latitude'],
                  marker='x', color='red', s=30, alpha=0.6)
    else:
        mask = db2 == cid
        ax.scatter(df[mask]['pickup_longitude'], df[mask]['pickup_latitude'],
                  color=plt.cm.tab20(cid % 20), s=30, alpha=0.7)
ax.set_title(f'epsilon OPTIMAL ({epsilon2:.5f} rad = {epsilon2*6371:.1f} km)\n{n_clusters2} clusters, {n_bruit2} bruit ({n_bruit2/len(df)*100:.0f}%)', 
            fontsize=11, fontweight='bold', color='green')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)

# Cas 3: epsilon TROP GRAND
epsilon3 = 0.002
db3 = DBSCAN(eps=epsilon3, min_samples=5, metric='haversine').fit_predict(coords_radians)
n_clusters3 = len(set(db3)) - (1 if -1 in db3 else 0)
n_bruit3 = len(db3[db3 == -1])

ax = axes[1, 0]
for cid in sorted(set(db3)):
    if cid == -1:
        mask = db3 == cid
        ax.scatter(df[mask]['pickup_longitude'], df[mask]['pickup_latitude'],
                  marker='x', color='red', s=30, alpha=0.6)
    else:
        mask = db3 == cid
        ax.scatter(df[mask]['pickup_longitude'], df[mask]['pickup_latitude'],
                  color=plt.cm.tab20(cid % 20), s=30, alpha=0.7)
ax.set_title(f'epsilon TROP GRAND ({epsilon3:.5f} rad = {epsilon3*6371:.1f} km)\n{n_clusters3} clusters, {n_bruit3} bruit ({n_bruit3/len(df)*100:.0f}%)', 
            fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)

# Cas 4: min_samples TRES GRAND
db4 = DBSCAN(eps=epsilon2, min_samples=50, metric='haversine').fit_predict(coords_radians)
n_clusters4 = len(set(db4)) - (1 if -1 in db4 else 0)
n_bruit4 = len(db4[db4 == -1])

ax = axes[1, 1]
for cid in sorted(set(db4)):
    if cid == -1:
        mask = db4 == cid
        ax.scatter(df[mask]['pickup_longitude'], df[mask]['pickup_latitude'],
                  marker='x', color='red', s=30, alpha=0.6)
    else:
        mask = db4 == cid
        ax.scatter(df[mask]['pickup_longitude'], df[mask]['pickup_latitude'],
                  color=plt.cm.tab20(cid % 20), s=30, alpha=0.7)
ax.set_title(f'min_samples TRES GRAND (50 au lieu de 5)\n{n_clusters4} clusters, {n_bruit4} bruit ({n_bruit4/len(df)*100:.0f}%)', 
            fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nCOMPARAISON:')
print(f'   epsilon={epsilon1} (trop petit):  {n_clusters1} clusters, {n_bruit1/len(df)*100:.1f}% bruit')
print(f'   epsilon={epsilon2} (optimal):  {n_clusters2} clusters, {n_bruit2/len(df)*100:.1f}% bruit  <-- MEILLEUR')
print(f'   epsilon={epsilon3} (trop grand):  {n_clusters3} clusters, {n_bruit3/len(df)*100:.1f}% bruit')
print(f'   min_samples=50:      {n_clusters4} clusters, {n_bruit4/len(df)*100:.1f}% bruit')

# CONCLUSION

In [ ]:
print('\n' + '='*80)
print('RESUME ET CONCLUSIONS')
print('='*80)

print(f'''
DOCUMENT: CE QUE NOUS AVONS APPRIS

1. PARTIE 1 - PREPARATION:
   OK Donnees NYC Taxi: {len(df):,} pickups apres nettoyage
   OK Variables: latitude, longitude (en radians pour Haversine)
   OK Bounds NYC: lat 40.5-40.95, lon -74.3 a -73.7

2. PARTIE 2 - DISTANCE HAVERSINE:
   OK Distance euclidienne = 85km ERROR sur NY!
   OK Haversine = vraie distance sur sphere
   OK OBLIGATOIRE pour GPS avec DBSCAN

3. PARTIE 3 - PARAMETRES:
   OK Epsilon en RADIANS (0.0005 rad = 3.2 km)
   OK min_samples ajuste selon densite (5-10 typique NYC)
   OK 2D = simple a tuner vs. haute dimension

4. PARTIE 4 - METHODE k-DISTANCE:
   OK Courbe k-distance identifie coude = epsilon
   OK Coude = separation clusters / bruit
   OK Epsilon optimal = {epsilon:.5f} rad (~{epsilon*6371:.1f} km)

5. PARTIE 5 - DBSCAN APPLICATION:
   OK {n_clusters} clusters identifies
   OK {n_bruit} points bruit ({n_bruit/len(df)*100:.1f}%)
   OK Hotspots: Manhattan, JFK, Brooklyn, LaGuardia

6. PARTIE 6 - VISUALISATION:
   OK Clusters sur carte NYC bien separes
   OK Centres identifies par zone
   OK Points bruit = trajets aberrants/isoles

7. PARTIE 7 - INTERPRETATION METIER:
   OK Zones aeroportuaires: JFK, LaGuardia
   OK Zones touristique/affaires: Manhattan Midtown
   OK Zones residentielles: Brooklyn
   OK Zones affaires: Financial District

8. PARTIE 8 - ANALYSE CRITIQUE:
   OK epsilon petit => tout bruit, structures perdues
   OK epsilon grand => 1 mega-cluster, inutile
   OK min_samples grand => clusters hyper-purs
   OK Limite: densites heterogenes mal gerees
   OK Alternative: HDBSCAN pour robustesse

BON TRAVAIL!
''')

print('='*80)